# Setting up the environment

## Importing libraries

In [1]:
import requests
import pandas as pd
import pprint
import time
import json

## Loading data from a JSON file

To load the data back into a DataFrame, use `pd.read_json()`. If your JSON was saved with `orient='records'`, loading it back in this way will correctly create your DataFrame.

In [15]:
# Movie data
movie_file = 'movies.json'
df = pd.read_json(movie_file, orient='records')
print(f'DataFrame successfully loaded from {movie_file}')

# Genre data
genre_file = 'genres.json'
with open(genre_file, 'r') as f:
    genre_map = json.load(f)
print(f'DataFrame successfully loaded from {genre_file}')

DataFrame successfully loaded from movies.json
DataFrame successfully loaded from genres.json


In [16]:
display(df.head())
pprint.pprint(genre_map)

,id,title,overview,genre_ids,keywords
0,1007757,Swapped,"A small woodland creature and a majestic bird,...","[12, 16, 10751, 14]","[wolf, buddy, forest-fire, woodlands, forest-l..."
1,278,The Shawshank Redemption,Imprisoned in the 1940s for the double murder ...,"[18, 80]","[prison, friendship, police-brutality, corrupt..."
2,238,The Godfather,"Spanning the years 1945 to 1955, a chronicle o...","[18, 80]","[based-on-novel-or-book, loss-of-loved-one, lo..."
3,687163,Project Hail Mary,Science teacher Ryland Grace wakes up on a spa...,"[878, 12]","[friendship, coma, based-on-novel-or-book, bra..."
4,936075,Michael,"The story of Michael Jackson, one of the most ...","[10402, 18]","[child-abuse, sibling-relationship, 1970s, abu..."


{'10402': 'Music',
 '10749': 'Romance',
 '10751': 'Family',
 '10752': 'War',
 '10770': 'TV Movie',
 '12': 'Adventure',
 '14': 'Fantasy',
 '16': 'Animation',
 '18': 'Drama',
 '27': 'Horror',
 '28': 'Action',
 '35': 'Comedy',
 '36': 'History',
 '37': 'Western',
 '53': 'Thriller',
 '80': 'Crime',
 '878': 'Science Fiction',
 '9648': 'Mystery',
 '99': 'Documentary'}


# DataFrame Preprocessing

## Generating keywords text

In [17]:
df['keywords_text'] = df['keywords'].apply(lambda x: " ".join(x))
display(df[['keywords', 'keywords_text']].head())

,keywords,keywords_text
0,"[wolf, buddy, forest-fire, woodlands, forest-l...",wolf buddy forest-fire woodlands forest-lore b...
1,"[prison, friendship, police-brutality, corrupt...",prison friendship police-brutality corruption ...
2,"[based-on-novel-or-book, loss-of-loved-one, lo...",based-on-novel-or-book loss-of-loved-one love-...
3,"[friendship, coma, based-on-novel-or-book, bra...",friendship coma based-on-novel-or-book bravery...
4,"[child-abuse, sibling-relationship, 1970s, abu...",child-abuse sibling-relationship 1970s abusive...


## Mapping genre-ids to genre-names

In [18]:
# Function to map genre IDs to genre names 
def genre_names_from_id(genre_id_ls):
	genre_name_ls = []

	for id in genre_id_ls:
		str_id = str(id)
		genre_name = genre_map[str_id].lower().replace(" ", "-")
		genre_name_ls.append(genre_name)

	genre_names = " ".join(genre_name_ls)
	return genre_names

# Applying function on data frame 
df['genre_names'] = df['genre_ids'].apply(genre_names_from_id)
df[['genre_ids', 'genre_names']].head()

,genre_ids,genre_names
0,"[12, 16, 10751, 14]",adventure animation family fantasy
1,"[18, 80]",drama crime
2,"[18, 80]",drama crime
3,"[878, 12]",science-fiction adventure
4,"[10402, 18]",music drama


# Text Preprocessing

We apply the following preprocessing steps:


1. Lowercasing
2. Removing punctuations
3. Tokenization
4. Stopword removal
5. Stemming



## Lowercasing

We lowercase the title, overview and keywords columns

In [19]:
df['title'] = df['title'].str.lower()
df['overview'] = df['overview'].str.lower()

display(df[['title', 'overview', 'keywords_text', 'genre_names']].head())

,title,overview,keywords_text,genre_names
0,swapped,"a small woodland creature and a majestic bird,...",wolf buddy forest-fire woodlands forest-lore b...,adventure animation family fantasy
1,the shawshank redemption,imprisoned in the 1940s for the double murder ...,prison friendship police-brutality corruption ...,drama crime
2,the godfather,"spanning the years 1945 to 1955, a chronicle o...",based-on-novel-or-book loss-of-loved-one love-...,drama crime
3,project hail mary,science teacher ryland grace wakes up on a spa...,friendship coma based-on-novel-or-book bravery...,science-fiction adventure
4,michael,"the story of michael jackson, one of the most ...",child-abuse sibling-relationship 1970s abusive...,music drama


## Removing punctuations

In [20]:
import string
string.punctuation

'!"#$%&\'()*+,-./:;<=>?@[\\]^_`{|}~'

In [21]:
punctuations = string.punctuation

# Function to remove punctuation from a given text
def exclude_punc(text):
    for char in text:
        if char in punctuations:
            text = text.replace(char, '')

    return text

df['overview'] = df['overview'].apply(exclude_punc)
df['overview'].head()

0    a small woodland creature and a majestic bird ...
1    imprisoned in the 1940s for the double murder ...
2    spanning the years 1945 to 1955 a chronicle of...
3    science teacher ryland grace wakes up on a spa...
4    the story of michael jackson one of the most i...
Name: overview, dtype: object

## Downloading NLTK modules

We download `punkt` and `punkt_tab` for tokenization and check whether the installation is succesful

In [ ]:
import nltk
import os

# Installing modules
nltk.download('punkt', quiet=False)
nltk.download("punkt_tab")

# Checking for successful installation 
for path in nltk.data.path:
	tokenizer_path = os.path.join(path, "tokenizers")
    
	if os.path.exists(tokenizer_path):
		print("Found:", tokenizer_path)
		print(os.listdir(tokenizer_path))


[nltk_data] Downloading package punkt to C:\Users\ARYA
[nltk_data]     BASAK\AppData\Roaming\nltk_data...
[nltk_data]   Unzipping tokenizers\punkt.zip.
[nltk_data] Downloading package punkt_tab to C:\Users\ARYA
[nltk_data]     BASAK\AppData\Roaming\nltk_data...
[nltk_data]   Unzipping tokenizers\punkt_tab.zip.


Found: C:\Users\ARYA BASAK\AppData\Roaming\nltk_data\tokenizers
['punkt', 'punkt.zip', 'punkt_tab', 'punkt_tab.zip']


## Stopword removal

In [22]:
nltk.download('stopwords', quiet=False)
from nltk.corpus import stopwords
stopwords.words('english')

[nltk_data] Downloading package stopwords to C:\Users\ARYA
[nltk_data]     BASAK\AppData\Roaming\nltk_data...
[nltk_data]   Unzipping corpora\stopwords.zip.


['a',
 'about',
 'above',
 'after',
 'again',
 'against',
 'ain',
 'all',
 'am',
 'an',
 'and',
 'any',
 'are',
 'aren',
 "aren't",
 'as',
 'at',
 'be',
 'because',
 'been',
 'before',
 'being',
 'below',
 'between',
 'both',
 'but',
 'by',
 'can',
 'couldn',
 "couldn't",
 'd',
 'did',
 'didn',
 "didn't",
 'do',
 'does',
 'doesn',
 "doesn't",
 'doing',
 'don',
 "don't",
 'down',
 'during',
 'each',
 'few',
 'for',
 'from',
 'further',
 'had',
 'hadn',
 "hadn't",
 'has',
 'hasn',
 "hasn't",
 'have',
 'haven',
 "haven't",
 'having',
 'he',
 "he'd",
 "he'll",
 'her',
 'here',
 'hers',
 'herself',
 "he's",
 'him',
 'himself',
 'his',
 'how',
 'i',
 "i'd",
 'if',
 "i'll",
 "i'm",
 'in',
 'into',
 'is',
 'isn',
 "isn't",
 'it',
 "it'd",
 "it'll",
 "it's",
 'its',
 'itself',
 "i've",
 'just',
 'll',
 'm',
 'ma',
 'me',
 'mightn',
 "mightn't",
 'more',
 'most',
 'mustn',
 "mustn't",
 'my',
 'myself',
 'needn',
 "needn't",
 'no',
 'nor',
 'not',
 'now',
 'o',
 'of',
 'off',
 'on',
 'once',
 'on

In [25]:
stop_words = stopwords.words('english')

# Function to remove stop words from a text
def remove_stopwords(text):
    words = text.split()
    return [word for word in words if word not in stop_words]

df['overview'] = df['overview'].apply(remove_stopwords)
display(df['overview'].head())

0    [small, woodland, creature, majestic, bird, tw...
1    [imprisoned, 1940s, double, murder, wife, love...
2    [spanning, years, 1945, 1955, chronicle, ficti...
3    [science, teacher, ryland, grace, wakes, space...
4    [story, michael, jackson, one, influential, ar...
Name: overview, dtype: object

## Stemming

In [26]:
from nltk.stem import PorterStemmer
ps = PorterStemmer()

ps.stem('walked')

'walk'

In [27]:
# Function to apply stemming 
def apply_stemming(words):
    return [ps.stem(word) for word in words]

df['overview'] = df['overview'].apply(apply_stemming)
display(df['overview'].head())

0    [small, woodland, creatur, majest, bird, two, ...
1    [imprison, 1940, doubl, murder, wife, lover, u...
2    [span, year, 1945, 1955, chronicl, fiction, it...
3    [scienc, teacher, ryland, grace, wake, spacesh...
4    [stori, michael, jackson, one, influenti, arti...
Name: overview, dtype: object

## Tokenization

In [28]:
from nltk.tokenize import word_tokenize

df['overview'] = df['overview'].apply(lambda x: " ".join(x))
df['tags'] = df['title'] + " " + df['overview'] + " " + df['genre_names']  + " " + df['keywords_text']
df['tokenized_tags'] = df['tags'].apply(word_tokenize)
display(df[['tags', 'tokenized_tags']].head())

,tags,tokenized_tags
0,swapped small woodland creatur majest bird two...,"[swapped, small, woodland, creatur, majest, bi..."
1,the shawshank redemption imprison 1940 doubl m...,"[the, shawshank, redemption, imprison, 1940, d..."
2,the godfather span year 1945 1955 chronicl fic...,"[the, godfather, span, year, 1945, 1955, chron..."
3,project hail mary scienc teacher ryland grace ...,"[project, hail, mary, scienc, teacher, ryland,..."
4,michael stori michael jackson one influenti ar...,"[michael, stori, michael, jackson, one, influe..."


# Feature Extaction/Text Representation

We use Tf-Idf technique for representing words as vectors

## Concatenating stemmed review tokens

In [29]:
from sklearn.feature_extraction.text import TfidfVectorizer

df['tokenized_tags_str'] = df['tokenized_tags'].apply(lambda tokens: ' '.join(tokens))
display(df['tokenized_tags_str'].head())

0    swapped small woodland creatur majest bird two...
1    the shawshank redemption imprison 1940 doubl m...
2    the godfather span year 1945 1955 chronicl fic...
3    project hail mary scienc teacher ryland grace ...
4    michael stori michael jackson one influenti ar...
Name: tokenized_tags_str, dtype: object

## Applying Tf-Idf

In [30]:
tfidf = TfidfVectorizer()
tfidf_matrix = tfidf.fit_transform(df['tokenized_tags_str'])
tfidf_array = tfidf_matrix.toarray()

print(tfidf_array)

[[0. 0. 0. ... 0. 0. 0.]
 [0. 0. 0. ... 0. 0. 0.]
 [0. 0. 0. ... 0. 0. 0.]
 ...
 [0. 0. 0. ... 0. 0. 0.]
 [0. 0. 0. ... 0. 0. 0.]
 [0. 0. 0. ... 0. 0. 0.]]


In [31]:
tfidf_array.shape

(10000, 31058)

In [32]:
tfidf.get_feature_names_out()

array(['00', '000', '007', ..., '살인', '스릴러', '인스팅트'], dtype=object)

# Making the recommendation system

## Calculating the similarity matrix between movies

In [33]:
from sklearn.metrics.pairwise import cosine_similarity
sim_matrix = cosine_similarity(tfidf_matrix, tfidf_matrix)

print(f"Shape of similarity matrix: {sim_matrix.shape}")
print(f"First 5x5 segment")
print(sim_matrix[:5, :5])

Shape of similarity matrix: (10000, 10000)
First 5x5 segment
[[1.         0.00198461 0.00528526 0.03529896 0.00802081]
 [0.00198461 1.         0.03573174 0.03624054 0.00829541]
 [0.00528526 0.03573174 1.         0.02740921 0.05245045]
 [0.03529896 0.03624054 0.02740921 1.         0.        ]
 [0.00802081 0.00829541 0.05245045 0.         1.        ]]


## Recommendation function

In [34]:
# Function to calculate similarity between two genres
def genre_similarity(query_genre, recommended_genre):
    q_set = set(query_genre)
    g_set = set(recommended_genre)

    if len(q_set | g_set) == 0:
        return 0

    return len(q_set & g_set)/len(q_set | g_set)

# Movie Recommendation Function
def get_recommendations(title, num_recommendations):
    try:
        index = df[df['title'] == title.lower()].index[0]
        genre = df['genre_ids'].iloc[index]
    except Exception:
        print(f"Movie {title} not found in the dataset")
        return

    # Similarity scores for the given movie
    sim_scores = list(enumerate(sim_matrix[index]))
    sim_scores = [(i, score) for (i, score) in sim_scores if i!=index]    # Removing the movie itself from the recommendations
    sorted_scores = sorted(sim_scores, key=lambda x: x[1], reverse=True)

    # Movies which are similar to the given one
    top_indices = [i[0] for i in sorted_scores[:num_recommendations+1]]
    top_movies = df['title'].iloc[top_indices].tolist()

    # Calculating similarity scroe based on overlapping genres
    recommended_genres = df['genre_ids'].iloc[top_indices].tolist()
    scores = [genre_similarity(genre, rgenre) for rgenre in recommended_genres]

    return top_movies, scores


## Applying the recommendation function

In [35]:
input_movies = ['swapped', 'the godfather', 'project hail mary']

for movie in input_movies:
    rec_movies, scores = get_recommendations(movie, 5)
    print(f"Input movie: {movie}")
    print(f"Recommendations: {rec_movies}")
    print(f"Similarity scores (based on genre overlap): {scores}")
    print()

Input movie: swapped
Recommendations: ['the angry birds movie', 'the wolf and the lion', 'for the birds', 'epic', 'family switch', 'flow']
Similarity scores (based on genre overlap): [0.6, 0.4, 0.4, 1.0, 0.4, 1.0]

Input movie: the godfather
Recommendations: ['the godfather part ii', 'the godfather part iii', 'the mafia kills only in summer', 'the alto knights', 'mafia mamma', 'the traitor']
Similarity scores (based on genre overlap): [1.0, 0.6666666666666666, 0.5, 0.6666666666666666, 0.25, 0.6666666666666666]

Input movie: project hail mary
Recommendations: ['sunshine', 'spaceman', 'gravity', 'red planet', 'rogue one: a star wars story', 'zathura: a space adventure']
Similarity scores (based on genre overlap): [0.3333333333333333, 0.6666666666666666, 0.25, 0.25, 0.6666666666666666, 0.6666666666666666]

